In [5]:
# CELL 1: Install packages

!pip install langchain langchain-community langchain-groq chromadb gradio pypdf sentence-transformers -q

print("Done")

Done


In [6]:
# CELL 2: API Key

import os
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("Loaded from secrets")
except:
    key = input("Paste Groq API key: ")
    os.environ["GROQ_API_KEY"] = key
    print("Key set")

Paste Groq API key: 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
Key set


In [7]:
# CELL 3: Create playbook

text = """Sales Playbook

Pricing:
- Basic: $49/month
- Pro: $99/month

Objection Handling:
If customer says "too expensive", say: Our customers save 15 hours per week.

Features:
- 24/7 support
- Free migration"""

with open("playbook.txt", "w") as f:
    f.write(text)

print("File created")

File created


In [8]:
# CELL 4: Simple RAG Chatbot

import os
import gradio as gr

# Simple document loading
with open("playbook.txt", "r") as f:
    document_text = f.read()

# Simple split into chunks
chunks = []
chunk_size = 300
for i in range(0, len(document_text), chunk_size):
    chunks.append(document_text[i:i+chunk_size])

print(f"Created {len(chunks)} chunks")

# Simple embedding function (using sentence-transformers)
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = model.encode(chunks)

# Simple search function
def search(query, k=2):
    query_embedding = model.encode([query])[0]
    similarities = np.dot(chunk_embeddings, query_embedding)
    top_indices = np.argsort(similarities)[-k:][::-1]
    return [chunks[i] for i in top_indices]

# Setup LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

# Chat function
def chat(message, history):
    # Search for relevant chunks
    relevant_chunks = search(message, k=2)
    context = "\n\n".join(relevant_chunks)

    # Create prompt
    prompt = f"""Answer based ONLY on this context. If answer not there, say "I cannot find that in the playbook."

Context:
{context}

Question: {message}

Answer:"""

    # Get response
    response = llm.invoke(prompt)
    return response.content

# Launch interface
demo = gr.ChatInterface(
    fn=chat,
    title="Sales Playbook RAG Chatbot",
    description="Ask questions about the playbook"
)

demo.launch(share=True)

Created 1 chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://de72abade933089a95.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
